# Why Word Order Matters, and What Bags of Words Cannot See

**Computational Text Analysis Pack · Module 24 — Sequence Models**

*Technique:* The limits of order-free representations; recurrence as the first fix

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/12-neural-foundations/24a_why_order_matters.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Every representation this course has used — frequency lists, TF-IDF, MFW
profiles, even the embedding-average model in module 13 — throws word order
away. That has been fine, because the questions were about *what* a text is
about or *who* wrote it, and those survive shuffling.

Some questions do not. This notebook finds the boundary.

## The tooling

**Keras** (on TensorFlow) for the models here — its `Sequential` API is the most
readable way to state an architecture, which is why teaching material reaches
for it. **PyTorch** is the research default and what almost every paper you read
will use; the concepts transfer directly, only the syntax differs.

Both are installed here. Nothing in these two modules needs a GPU: the models
are a few hundred thousand parameters and train on CPU in under a minute.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# The demonstration: sentences that are identical as bags of words.
PAIRS = [
    ("the government criticised the newspaper",
     "the newspaper criticised the government"),
    ("the doctor treated the patient who called the nurse",
     "the nurse treated the patient who called the doctor"),
    ("aid reached the village before the flood",
     "the flood reached the village before aid"),
]
for a, b in PAIRS:
    ca, cb = Counter(tokenise(a)), Counter(tokenise(b))
    print(f"  A: {a}")
    print(f"  B: {b}")
    print(f"     identical as bags of words: {ca == cb}")
    print()
print("Every method in modules 2-12 scores these pairs as the same text.")
print("They mean opposite things. Word order is not decoration here — it is")
print("the entire content of the difference.")

## Does it matter for real tasks?

Honest answer: usually less than you would expect. Test it rather than assume —
shuffle the words in every document and see how much accuracy a classifier
loses.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(0)
sub = df[df["n_words"] >= 30].sample(n=3000, random_state=1).reset_index(drop=True)

def shuffled(text):
    toks = tokenise(text)
    rng.shuffle(toks)
    return " ".join(toks)

sub["scrambled"] = sub["text"].map(shuffled)
tr, te = train_test_split(np.arange(len(sub)), test_size=0.3, random_state=0,
                          stratify=sub["category"])

for col, label in [("text", "original word order"), ("scrambled", "words shuffled")]:
    v = TfidfVectorizer(token_pattern=r"\w+", min_df=3, max_features=10000)
    A = v.fit_transform(sub[col].iloc[tr]); B = v.transform(sub[col].iloc[te])
    m = LogisticRegression(max_iter=1000).fit(A, sub["category"].iloc[tr])
    print(f"   {label:<24}{accuracy_score(sub['category'].iloc[te], m.predict(B)):.3f}")
print()
print("Barely any difference. For TOPIC classification, order carries almost")
print("nothing — which is why bag-of-words survived fifty years. Keep that in")
print("mind before reaching for a sequence model: check whether your task")
print("actually needs one.")

## Where recurrence comes in

A recurrent network reads left to right, carrying a hidden state forward. The
state is the model's memory of what it has read, and it is what lets *the
government criticised the newspaper* differ from its reverse.

The mechanism is small enough to write out, so here it is: no library, just the
loop.

In [ ]:
# A recurrent step, by hand. h_t = tanh(W_x x_t + W_h h_{t-1} + b)
rng = np.random.default_rng(3)
DIM = 4
Wx = rng.normal(0, .5, (DIM, DIM)); Wh = rng.normal(0, .5, (DIM, DIM))
b = np.zeros(DIM)
vocab = {w: rng.normal(0, 1, DIM) for w in
         ["the", "government", "criticised", "newspaper"]}

def run_rnn(sentence):
    h = np.zeros(DIM)
    states = []
    for w in tokenise(sentence):
        x = vocab.get(w, np.zeros(DIM))
        h = np.tanh(Wx @ x + Wh @ h + b)
        states.append((w, h.copy()))
    return states

for s in ["the government criticised the newspaper",
          "the newspaper criticised the government"]:
    states = run_rnn(s)
    print(f"{s}\n   final state: {np.round(states[-1][1], 3)}")
print()
print("Same words, different final state — because the state depends on the")
print("order they arrived in. That is the whole idea; LSTMs and GRUs are")
print("elaborations that stop the state from decaying over long distances.")

### 🔧 Your turn 1

Shuffle only *within* sentences rather than across the whole document, and re-run the classifier comparison. Does the result change?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Hardly at all, which sharpens the point: topic information lives in *which*
words are present, at document scale, and no amount of local reordering touches
that. Order matters for tasks where the unit of meaning is the clause —
sentiment with negation, relation extraction, anything about who did what to
whom. Matching the model to the unit of meaning your question lives at is the
decision; reaching for the more powerful architecture is not.

</details>